In [17]:
import pandas as pd
import sqlite3
import re
import os

In [18]:
data_dir = 'data'
db_name = 'Cleaned_market_data.db'

valid_order_statuses = {'pending', 'completed', 'cancelled'}

def is_email_valid(email):
    if pd.isna(email):
        return False
    regex = r'^[\w\.-]+@[\w\.-]+\.\w+$'
    return bool(re.match(regex, str(email)))

def run_pipeline():
    print("Starting ETL process...")
    audit_stats = []

    print("Extracting raw data...")
    customers = pd.read_csv(os.path.join(data_dir, 'customers.csv'))
    products = pd.read_csv(os.path.join(data_dir, 'products.csv'))
    orders = pd.read_csv(os.path.join(data_dir, 'orders.csv'))
    order_items = pd.read_csv(os.path.join(data_dir, 'order_items.csv'))

    print('Cleaning Customers table...')
    c_intitial = len(customers)

    customers = customers.dropna(subset=['customer_id'])
    customers = customers.drop_duplicates(subset = ['customer_id'])
    customers['email'] = customers['email'].apply(lambda x: x if is_email_valid(x) else 'unknown@domain.com')
    customers['created_at'] = pd.to_datetime(customers['created_at'], errors = 'coerce')
    customers = customers.dropna(subset = ['created_at'])

    audit_stats.append({'table_name': 'customers',
                        'initial_rows': c_intitial,
                        'final_rows': len(customers),
                        'dropped_rows': c_intitial - len(customers)
                        })

    print('Cleaning Products table...')
    p_initial = len(products)

    products = products.dropna(subset=['product_id'])
    products = products.drop_duplicates(subset=['product_id'])
    products['price'] = pd.to_numeric(products['price'], errors='coerce')
    products = products[products['price'] > 0] 
    invalid_data = ['test', 'dummy', 'unknown', 'n/a', 'nan']

    def is_invalid(text):
        if pd.isna(text):
            return True
        
        text_str = str(text).lower().strip()
        
        for pattern in invalid_data:
            if pattern in text_str:
                return True
        return False
 
    products = products[
        ~products['name'].apply(is_invalid) & 
        ~products['category'].apply(is_invalid)
    ]
    
    products['category'] = products['category'].apply(lambda x: str(x).strip().title())

    audit_stats.append({'table_name': 'products', 
                        'initial_rows': p_initial, 
                        'final_rows': len(products), 
                        'dropped_rows': p_initial - len(products)
                        })

    print('CLeaning Orders table...')
    o_initial = len(orders)

    orders = orders.dropna(subset = ['order_id', 'customer_id'])
    orders = orders.drop_duplicates(subset = ['order_id'])
    orders['created_at'] = pd.to_datetime(orders['created_at'], errors = 'coerce')
    orders = orders.dropna(subset = ['created_at'])

    orders['order_status'] = orders['order_status'].astype(str).str.strip().str.lower()
    orders = orders[orders['order_status'].isin(valid_order_statuses)]

    orders = orders[orders['customer_id'].isin(customers['customer_id'])]

    audit_stats.append({'table_name': 'orders', 
                        'initial_rows': o_initial, 
                        'final_rows': len(orders), 
                        'dropped_rows': o_initial - len(orders)
                        })

    print('Cleaning Order Items table...')
    oi_initial = len(order_items)

    order_items = order_items.dropna(subset = ['order_item_id', 'order_id', 'product_id'])
    order_items = order_items.drop_duplicates(subset = ['order_item_id'])
    order_items['quantity'] = pd.to_numeric(order_items['quantity'], errors = 'coerce')
    order_items = order_items[order_items['quantity'] > 0]

    order_items = order_items[order_items['order_id'].isin(orders['order_id'])]
    order_items = order_items[order_items['product_id'].isin(products['product_id'])]

    audit_stats.append({'table_name': 'order_items', 
                        'initial_rows': oi_initial, 
                        'final_rows': len(order_items), 
                        'dropped_rows': oi_initial - len(order_items)
                        })
    
    df_audit = pd.DataFrame(audit_stats)

    print("Loading Data to SQLite database...")
    with sqlite3.connect(db_name) as conn:
        customers.to_sql('dim_customers', conn, if_exists = 'replace', index = False)
        products.to_sql('dim_products', conn, if_exists = 'replace', index = False)
        orders.to_sql('fct_orders', conn, if_exists = 'replace', index = False)
        order_items.to_sql('fct_order_items', conn, if_exists = 'replace', index = False)
    
    print(f"Data was uplouded to {db_name} database")

    return df_audit


In [19]:
run_pipeline()

Starting ETL process...
Extracting raw data...
Cleaning Customers table...
Cleaning Products table...
CLeaning Orders table...
Cleaning Order Items table...
Loading Data to SQLite database...
Data was uplouded to Cleaned_market_data.db database


,table_name,initial_rows,final_rows,dropped_rows
0,customers,357,352,5
1,products,155,150,5
2,orders,456,451,5
3,order_items,914,906,8


Підсумок ETL-процесу
1. Що та чому було очищено:
    Клієнти (customers): Видалено записи без ID та дублікати. Невалідні email замінено на unknown@domain.com, щоб зберегти історію замовлень для аналітики.
    Товари (products): Застосовано фільтрацію за "чорним списком" (test, dummy, unknown тощо) для назв та категорій. Видалено товари з ціною $\leq 0$, 
    Замовлення (orders): Стандартизовано статуси до нижнього регістру та залишено лише валідні (pending, completed, cancelled). Видалено замовлення без прив'язки до існуючого клієнта.
    Деталі замовлень (order_items): Видалено записи з від'ємною кількістю та перевірено цілісність зв'язків з таблицями замовлень і товарів.

2. Створені вихідні таблиці:
У базу даних Cleaned_market_data.db завантажено наступні таблиці:
    dim_customers — очищені дані про клієнтів.
    dim_products — каталог товарів без тестового сміття.
    fct_orders — реєстр замовлень із перевіреною цілісністю.
    fct_order_items — деталізація замовлень для аналізу продажів.

3. Після виконання пайплайну виводиться таблиця df_audit, яка демонструє обсяг відфільтрованих даних для кожної таблиці.